# Notebook 04 – Feature Engineering

## Objective

The objective of this notebook is to transform the cleaned dataset into a machine-learning-ready format.

The following preprocessing steps are performed:

- Feature selection
- Train, validation, and test split
- Numerical feature scaling
- Categorical feature encoding
- Preprocessing pipeline creation
- Saving preprocessing objects for model deployment

### Input

data/processed/cleaned_patient_dataset.csv

### Output

Encoded and scaled datasets ready for machine learning.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

import joblib

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [2]:
df = pd.read_csv(
    "../data/processed/cleaned_patient_dataset.csv"
)

print(df.shape)

df.head()

(5000, 29)


,Age,Gender,Height_cm,Weight_kg,BMI,Chronic_Disease,Blood_Pressure_Systolic,Blood_Pressure_Diastolic,Cholesterol_Level,Blood_Sugar_Level,...,Protein_Intake,Carbohydrate_Intake,Fat_Intake,Preferred_Cuisine,Food_Aversions,Recommended_Calories,Recommended_Protein,Recommended_Carbs,Recommended_Fats,Recommended_Meal_Plan
0,56,Other,163,66,24.84,No Disease,175,75,219,124,...,105,179,143,Western,NaN,2150,108,139,145,High-Protein Diet
1,69,Female,171,114,38.99,No Disease,155,72,208,72,...,69,315,75,Mediterranean,NaN,1527,74,266,80,Low-Carb Diet
2,46,Female,172,119,40.22,No Disease,137,101,171,145,...,183,103,148,Western,Sweet,2359,180,145,143,Low-Carb Diet
3,32,Female,197,118,30.41,No Disease,148,91,258,235,...,135,371,120,Western,NaN,2858,137,378,135,Low-Carb Diet
4,60,Female,156,109,44.79,Hypertension,160,109,260,248,...,167,298,48,Indian,Spicy,1937,166,317,56,Low-Carb Diet


Step 3 — Define Targets

Regression

In [3]:
y_regression = df["Recommended_Calories"]

Classification

In [4]:
y_classification = df["Recommended_Meal_Plan"]

Step 4 — Create Feature Sets

Model A (Full Features)

In [5]:
X_full = df.drop(
    columns=[
        "Recommended_Calories",
        "Recommended_Protein",
        "Recommended_Carbs",
        "Recommended_Fats",
        "Recommended_Meal_Plan"
    ]
)


print(X_full.shape)

(5000, 24)


Model B (Clinical Features Only)

In [6]:
X_clinical = X_full.drop(
    columns=[
        "Caloric_Intake",
        "Protein_Intake",
        "Carbohydrate_Intake",
        "Fat_Intake"
    ]
)

print(X_clinical.shape)

(5000, 20)


🔹 Step 5 — Split the Data

We'll split the data into:

Training
Validation
Testing

In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_full,
    y_classification,
    test_size=0.30,
    random_state=42,
    stratify=y_classification
)

In [8]:
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [9]:
print(X_train.shape)
print(X_validation.shape)
print(X_test.shape)

(3500, 24)
(750, 24)
(750, 24)


## Step 7 – Identify Feature Types

Machine learning preprocessing differs for numerical and categorical variables.

- Numerical features will be standardized.
- Categorical features will be one-hot encoded.

Separating these feature types allows us to build a reusable preprocessing pipeline.

In [10]:
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

numerical_features = X_train.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

Categorical Features:
['Gender', 'Chronic_Disease', 'Genetic_Risk_Factor', 'Allergies', 'Alcohol_Consumption', 'Smoking_Habit', 'Dietary_Habits', 'Preferred_Cuisine', 'Food_Aversions']

Numerical Features:
['Age', 'Height_cm', 'Weight_kg', 'BMI', 'Blood_Pressure_Systolic', 'Blood_Pressure_Diastolic', 'Cholesterol_Level', 'Blood_Sugar_Level', 'Daily_Steps', 'Exercise_Frequency', 'Sleep_Hours', 'Caloric_Intake', 'Protein_Intake', 'Carbohydrate_Intake', 'Fat_Intake']


## Step 8 – Build the Preprocessing Pipeline

A ColumnTransformer is used to apply different preprocessing steps to numerical and categorical features.

- Numerical features → StandardScaler
- Categorical features → OneHotEncoder

This approach ensures consistent preprocessing during both training and deployment.

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


## Step 9 – Apply Feature Transformation

The preprocessing pipeline is fitted using only the training dataset to prevent data leakage.

The same transformations are then applied to the validation and test datasets.

In [12]:
X_train_processed = preprocessor.fit_transform(X_train)

X_validation_processed = preprocessor.transform(X_validation)

X_test_processed = preprocessor.transform(X_test)

print("Feature transformation completed successfully.")

Feature transformation completed successfully.


In [13]:
print("Training Data Shape   :", X_train_processed.shape)
print("Validation Data Shape :", X_validation_processed.shape)
print("Testing Data Shape    :", X_test_processed.shape)

Training Data Shape   : (3500, 45)
Validation Data Shape : (750, 45)
Testing Data Shape    : (750, 45)


Step 11 – Save the Preprocessing Pipeline

In [14]:
import joblib

joblib.dump(
    preprocessor,
    "../models/saved_objects/preprocessor.pkl"
)

print("Preprocessor saved successfully.")

Preprocessor saved successfully.


Step 12 – Save Feature Names

In [15]:
feature_names = preprocessor.get_feature_names_out()

joblib.dump(
    feature_names,
    "../models/saved_objects/feature_columns.pkl"
)

print("Feature names saved successfully.")

Feature names saved successfully.


In [16]:
X_train.to_csv("../data/splits/X_train.csv", index=False)
X_validation.to_csv("../data/splits/X_validation.csv", index=False)
X_test.to_csv("../data/splits/X_test.csv", index=False)

y_train.to_csv("../data/splits/y_train.csv", index=False)
y_validation.to_csv("../data/splits/y_validation.csv", index=False)
y_test.to_csv("../data/splits/y_test.csv", index=False)

print("Dataset splits saved successfully.")

Dataset splits saved successfully.


# Feature Engineering Summary

The feature engineering process has been completed successfully.

## Tasks Performed

- Selected input features and target variables.
- Split the dataset into training, validation, and testing sets.
- Identified numerical and categorical features.
- Standardized numerical variables using StandardScaler.
- Encoded categorical variables using OneHotEncoder.
- Built a reusable preprocessing pipeline with ColumnTransformer.
- Saved the preprocessing pipeline and transformed feature names.
- Saved dataset splits for reproducibility.

The processed data is now ready for machine learning model training.

In [17]:
import json

feature_report = {
    "Training Shape": X_train_processed.shape,
    "Validation Shape": X_validation_processed.shape,
    "Testing Shape": X_test_processed.shape,
    "Numerical Features": numerical_features,
    "Categorical Features": categorical_features,
    "Total Processed Features": len(feature_names)
}

with open("../reports/metrics/feature_engineering_report.json", "w") as f:
    json.dump(feature_report, f, indent=4)

print("Feature engineering report saved successfully.")

Feature engineering report saved successfully.
